# Algorithmic trading research: XGBoost + TA-Lib + backtesting.py

A leakage-aware, reproducible baseline: **data -> TA-Lib features -> purged walk-forward XGBoost -> backtest with costs -> metrics vs buy-and-hold**.

## Dataset source
- **Default: Tiingo end-of-day API** (free account). Default ticker `SPY`, 2010-01-01 to 2024-12-31. The notebook uses Tiingo's split/dividend-adjusted open, high, low, close and volume.
  - **Get a token:** sign up at tiingo.com, then open Account -> API and copy your token. Free-tier limits are modest (about 50 requests an hour, 1,000 a day), which is plenty for a few tickers.
  - **Give the notebook the token** in any one of three ways: set the `TIINGO_API_KEY` environment variable, add it as a Colab Secret named `TIINGO_API_KEY`, or paste it when prompted. It is never written into the notebook or the saved CSV.
- The first run downloads the data and **saves it as a CSV in `data/`**, so you keep the dataset as a file and later runs don't call the API again.
- **No-key alternative:** `source="yfinance"` uses Yahoo Finance (unofficial, no API key).
- **Your own file:** `source="csv"` with `csv_path` pointing to a CSV with columns `Date, Open, High, Low, Close, Volume` (an `Adj Close` column is used to adjust prices if present).
- **Offline test:** `source="synthetic"` generates fake prices. It only checks that the notebook runs and must not be used to draw conclusions.

The goal is an honest answer to "does this signal survive costs out of sample?", not a strategy that beats the market. Run top to bottom.

In [1]:
# Colab / fresh environment: install dependencies (TA-Lib >= 0.6 ships binary wheels)
%pip install -q numpy pandas requests scikit-learn xgboost TA-Lib backtesting yfinance plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.8/190.8 kB 10.6 MB/s eta 0:00:00


## 1. Configuration
Everything you may want to change lives in this one cell.

In [2]:
import os, random, json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

os.environ["TQDM_DISABLE"] = "1"      # silence backtesting.py progress bars
import warnings
warnings.filterwarnings("ignore", message=".*Jupyter Notebook detected.*")

CFG = dict(
    seed=42,
    data=dict(
        source="tiingo",          # "tiingo" | "yfinance" | "csv" | "synthetic"
        ticker="SPY",
        start="2010-01-01",
        end="2024-12-31",             # pin the end date so the saved dataset and results reproduce
        csv_path="data/my_prices.csv",
        cache_dir="data",
        refresh=False,                # True = re-download even if the CSV exists
        max_ffill=3,
    ),
    features=dict(                    # all scale-free ratios / bounded oscillators
        return_lags=[1, 5, 10, 20], sma=[10, 50], ema=[12, 26], rsi=[14],
        macd=dict(fast=12, slow=26, signal=9), bbands=dict(period=20, nbdev=2.0),
        atr=[14], vol_window=20,
    ),
    target=dict(horizon=1),           # label: open[t+1+h] / open[t+1] - 1 > 0  (signal at close t, fill at next open)
    model=dict(
        params=dict(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, min_child_weight=5, reg_lambda=5.0, n_jobs=4),
        cv=dict(n_splits=5, test_size=None, max_train_size=None),   # None = expanding window
    ),
    backtest=dict(
        cash=100_000, commission_bps=5, slippage_bps=5,             # costs are per side
        enter_prob=0.52, exit_prob=0.50, size_frac=0.99,
        periods_per_year=252, risk_free_annual=0.0,
    ),
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG["seed"])
D, F, B = CFG["data"], CFG["features"], CFG["backtest"]
H = CFG["target"]["horizon"]

## 2. Data ingestion
Fetch (or read), sanitize and save the dataset. Sanitizing sorts, removes duplicate dates, drops non-positive prices, forward-fills short gaps, and forces `High >= max(O,C)` and `Low <= min(O,C)`.

In [3]:
PRICE_COLS = ["Open", "High", "Low", "Close"]
OHLCV = PRICE_COLS + ["Volume"]

def sanitize(df, max_ffill=3):
    n0 = len(df)
    df = df[OHLCV].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.replace([np.inf, -np.inf], np.nan)
    bad = (df[PRICE_COLS] <= 0).any(axis=1)
    df.loc[bad, PRICE_COLS] = np.nan
    df[PRICE_COLS] = df[PRICE_COLS].ffill(limit=max_ffill)
    df["Volume"] = df["Volume"].fillna(0.0)
    df = df.dropna(subset=PRICE_COLS)
    df["High"] = df[PRICE_COLS].max(axis=1)
    df["Low"] = df[PRICE_COLS].min(axis=1)
    if len(df) < n0:
        print(f"sanitize dropped {n0 - len(df)} of {n0} rows")
    return df

def fetch_yfinance(ticker, start, end):
    import yfinance as yf
    df = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True, actions=False)
    if df.empty:
        raise ValueError(f"no data returned for {ticker} ({start} -> {end}); check the ticker, dates and internet access")
    df.index = pd.DatetimeIndex(df.index).tz_localize(None).normalize()
    return df[OHLCV]

def get_tiingo_token():
    tok = os.environ.get("TIINGO_API_KEY")
    if tok:
        return tok
    try:                                   # Colab Secrets
        from google.colab import userdata
        tok = userdata.get("TIINGO_API_KEY")
        if tok:
            return tok
    except Exception:
        pass
    import getpass
    return getpass.getpass("Tiingo API token (free at tiingo.com; not saved anywhere): ").strip()

def fetch_tiingo(ticker, start, end):
    import requests
    r = requests.get(
        f"https://api.tiingo.com/tiingo/daily/{ticker.lower()}/prices",
        params={"startDate": start, "endDate": end, "format": "json", "resampleFreq": "daily"},
        headers={"Authorization": f"Token {get_tiingo_token()}", "Content-Type": "application/json"},
        timeout=30,
    )
    if r.status_code != 200:
        raise RuntimeError(f"Tiingo returned HTTP {r.status_code}: {r.text[:200]} "
                           "(401/403 = bad token, 404 = unknown ticker, 429 = free-tier request limit reached)")
    rows = r.json()
    if not rows:
        raise ValueError(f"no data returned for {ticker} ({start} -> {end})")
    raw = pd.DataFrame(rows)
    idx = pd.DatetimeIndex(pd.to_datetime(raw["date"], utc=True).dt.tz_localize(None).dt.normalize())
    # adjusted fields: consistent for splits and dividends
    return pd.DataFrame({
        "Open": raw["adjOpen"].to_numpy(), "High": raw["adjHigh"].to_numpy(),
        "Low": raw["adjLow"].to_numpy(), "Close": raw["adjClose"].to_numpy(),
        "Volume": raw["adjVolume"].to_numpy(),
    }, index=idx)

def read_csv_prices(path):
    raw = pd.read_csv(path)
    raw.columns = [str(c).strip().lower().replace(" ", "_") for c in raw.columns]
    date_col = "date" if "date" in raw.columns else raw.columns[0]
    idx = pd.to_datetime(raw.pop(date_col), utc=True).dt.tz_localize(None).dt.normalize()
    raw.index = pd.DatetimeIndex(idx)
    factor = raw["adj_close"] / raw["close"] if "adj_close" in raw.columns else 1.0
    out = pd.DataFrame(index=raw.index)
    for c in PRICE_COLS:
        out[c] = raw[c.lower()] * factor
    out["Volume"] = raw["volume"] if "volume" in raw.columns else 0.0
    return out

def make_synthetic(start, end, seed=0):
    # regime-switching vol + weak AR(1) drift. Plumbing test only.
    idx = pd.bdate_range(start, end)
    n = len(idx)
    rng = np.random.default_rng(seed)
    vol = np.where(np.sin(np.arange(n) / 250 * 2 * np.pi) > 0.5, 0.02, 0.008)
    r = 0.0003 + vol * rng.standard_normal(n)
    for t in range(1, n):
        r[t] += 0.05 * r[t - 1]
    close = 100 * np.exp(np.cumsum(r))
    open_ = np.r_[close[0], close[:-1]] * (1 + 0.002 * rng.standard_normal(n))
    high = np.maximum(open_, close) * (1 + np.abs(0.003 * rng.standard_normal(n)))
    low = np.minimum(open_, close) * (1 - np.abs(0.003 * rng.standard_normal(n)))
    volume = np.exp(15 + 0.3 * rng.standard_normal(n))
    return pd.DataFrame({"Open": open_, "High": high, "Low": low, "Close": close, "Volume": volume}, index=idx)

def load_prices(d, seed):
    src = d["source"]
    if src == "synthetic":
        return sanitize(make_synthetic(d["start"], d["end"], seed), d["max_ffill"])
    if src == "csv":
        return sanitize(read_csv_prices(d["csv_path"]), d["max_ffill"]).loc[d["start"]:d["end"]]
    if src in ("tiingo", "yfinance"):
        path = Path(d["cache_dir"]) / f"{src}_{d['ticker']}_{d['start']}_{d['end']}.csv"
        if path.exists() and not d["refresh"]:
            print("loading saved dataset:", path)
            return sanitize(read_csv_prices(path), d["max_ffill"])
        fetch = fetch_tiingo if src == "tiingo" else fetch_yfinance
        df = sanitize(fetch(d["ticker"], d["start"], d["end"]), d["max_ffill"])
        path.parent.mkdir(parents=True, exist_ok=True)
        df.rename_axis("Date").to_csv(path)
        print(f"downloaded from {src} and saved dataset to:", path)
        return df
    raise ValueError(f"unknown source: {src}")

prices = load_prices(D, CFG["seed"])
print(f"{D['source']} | {len(prices)} rows | {prices.index[0].date()} -> {prices.index[-1].date()}")
display(prices.tail())

fig = go.Figure(go.Scatter(x=prices.index, y=prices["Close"], name="Close (adjusted)"))
fig.update_layout(title=f"{D['ticker']} adjusted close", height=320, margin=dict(t=40, b=10))
fig.show()

Tiingo API token (free at tiingo.com; not saved anywhere): ··········
downloaded from tiingo and saved dataset to: data/tiingo_SPY_2010-01-01_2024-12-31.csv
tiingo | 3774 rows | 2010-01-04 -> 2024-12-31


,Open,High,Low,Close,Volume
date,,,,,
2024-12-24,584.590269,589.768668,584.011622,589.729437,33160097
2024-12-26,587.964074,590.886731,586.573850,589.768668,41338891
2024-12-27,586.041790,586.273346,579.396864,583.560473,64969310
2024-12-30,576.577480,580.353396,573.164445,576.901130,56578757
2024-12-31,578.553707,579.274465,573.174252,574.802309,57052654


## 3. Features and target
All TA-Lib features use data up to and including bar *t* only. The target matches how the backtester fills: signal at the close of *t*, trade at the open of *t+1*, exit at the open of *t+1+h*.

In [4]:
import talib

def build_features(df, fcfg):
    close = df["Close"].to_numpy(dtype="float64")
    high = df["High"].to_numpy(dtype="float64")
    low = df["Low"].to_numpy(dtype="float64")
    idx = df.index
    out = {}
    ret1 = df["Close"].pct_change()
    for n in fcfg.get("return_lags", []):
        out[f"ret_{n}"] = df["Close"].pct_change(n)
    for n in fcfg.get("sma", []):
        out[f"close_over_sma{n}"] = close / talib.SMA(close, timeperiod=n) - 1
    for n in fcfg.get("ema", []):
        out[f"close_over_ema{n}"] = close / talib.EMA(close, timeperiod=n) - 1
    for n in fcfg.get("rsi", []):
        out[f"rsi_{n}"] = talib.RSI(close, timeperiod=n)
    if fcfg.get("macd"):
        m = fcfg["macd"]
        macd, sig, hist = talib.MACD(close, fastperiod=m["fast"], slowperiod=m["slow"], signalperiod=m["signal"])
        out["macd_norm"], out["macd_signal_norm"], out["macd_hist_norm"] = macd / close, sig / close, hist / close
    if fcfg.get("bbands"):
        b = fcfg["bbands"]
        up, mid, lo = talib.BBANDS(close, timeperiod=b["period"], nbdevup=b["nbdev"], nbdevdn=b["nbdev"])
        out["bb_pctb"] = (close - lo) / (up - lo)
        out["bb_width"] = (up - lo) / mid
    for n in fcfg.get("atr", []):
        out[f"atr_{n}_norm"] = talib.ATR(high, low, close, timeperiod=n) / close
    if fcfg.get("vol_window"):
        w = fcfg["vol_window"]
        out[f"realized_vol_{w}"] = ret1.rolling(w).std()
        out[f"volume_ratio_{w}"] = df["Volume"] / df["Volume"].rolling(w).mean()
    feats = pd.DataFrame({k: pd.Series(np.asarray(v), index=idx) for k, v in out.items()}, index=idx)
    return feats.replace([np.inf, -np.inf], np.nan)

def make_target(df, horizon=1):
    entry = df["Open"].shift(-1)
    exit_ = df["Open"].shift(-(1 + horizon))
    fwd = exit_ / entry - 1
    y = (fwd > 0).astype(float).where(fwd.notna())
    return pd.DataFrame({"fwd_ret": fwd, "y": y})

feats = build_features(prices, F)
tgt = make_target(prices, H)
ds = feats.join(tgt).dropna()          # drops indicator warm-up rows and the last h+1 unlabeled bars
X, y, fwd = ds[feats.columns], ds["y"].astype(int), ds["fwd_ret"]
print(f"{X.shape[1]} features | {len(X)} labeled rows | share of up-moves: {y.mean():.3f}")
display(X.tail(3).T.round(4))

17 features | 3723 labeled rows | share of up-moves: 0.559


date,2024-12-24,2024-12-26,2024-12-27
ret_1,0.0111,0.0001,-0.0105
ret_5,-0.0016,0.0291,0.0186
ret_10,0.0008,-0.0068,-0.0121
ret_20,0.0097,0.0045,-0.0031
close_over_sma10,0.0068,0.0075,-0.0019
close_over_sma50,0.0189,0.0182,0.0069
close_over_ema12,0.0078,0.0066,-0.0034
close_over_ema26,0.0097,0.0090,-0.0015
rsi_14,55.9958,56.0310,49.3024
macd_norm,0.0019,0.0024,0.0019


## 4. Purged walk-forward XGBoost
Expanding-window `TimeSeriesSplit` with `gap = horizon + 1`, so training labels never overlap the test window. Every prediction produced here is out-of-sample, and the test blocks are contiguous so they can be backtested as one period.

In [5]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

def make_model(params, seed):
    base = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist", importance_type="gain")
    return XGBClassifier(**{**base, **params, "random_state": seed})

def walk_forward_predict(X, y, fwd_ret, model_factory, n_splits=5, gap=2, test_size=None, max_train_size=None):
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap, test_size=test_size, max_train_size=max_train_size)
    parts, rows, imps = [], [], []
    for k, (tr, te) in enumerate(tscv.split(X)):
        model = model_factory()
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        y_te = y.iloc[te]
        parts.append(pd.DataFrame({"p_up": p, "y": y_te.values, "fwd_ret": fwd_ret.iloc[te].values, "fold": k}, index=X.index[te]))
        rows.append({
            "fold": k,
            "train_start": X.index[tr[0]].date(), "train_end": X.index[tr[-1]].date(),
            "test_start": X.index[te[0]].date(), "test_end": X.index[te[-1]].date(),
            "train_end_pos": int(tr[-1]), "test_start_pos": int(te[0]),
            "n_train": len(tr), "n_test": len(te),
            "accuracy": float(((p > 0.5) == y_te.values).mean()),
            "auc": float(roc_auc_score(y_te, p)) if y_te.nunique() > 1 else float("nan"),
        })
        imp = getattr(model, "feature_importances_", None)       # models without importances (momentum, LSTM) get zeros
        imps.append(pd.Series(np.zeros(X.shape[1]) if imp is None else np.asarray(imp), index=X.columns))
    oos = pd.concat(parts)
    importance = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
    return oos, pd.DataFrame(rows), importance

cv = CFG["model"]["cv"]
factory = lambda: make_model(CFG["model"]["params"], CFG["seed"])
oos, folds, importance = walk_forward_predict(
    X, y, fwd, factory, n_splits=cv["n_splits"], gap=H + 1,
    test_size=cv["test_size"], max_train_size=cv["max_train_size"])

display(folds.drop(columns=["train_end_pos", "test_start_pos"]).round(3))
acc = ((oos["p_up"] > 0.5).astype(int) == oos["y"]).mean()
base = max(oos["y"].mean(), 1 - oos["y"].mean())
auc = roc_auc_score(oos["y"], oos["p_up"])
print(f"OOS accuracy {acc:.3f} vs majority-class baseline {base:.3f} | OOS AUC {auc:.3f}")

,fold,train_start,train_end,test_start,test_end,n_train,n_test,accuracy,auc
0,0,2010-03-16,2012-08-28,2012-08-31,2015-02-20,621,620,0.511,0.499
1,1,2010-03-16,2015-02-18,2015-02-23,2017-08-07,1241,620,0.531,0.482
2,2,2010-03-16,2017-08-03,2017-08-08,2020-01-24,1861,620,0.524,0.513
3,3,2010-03-16,2020-01-22,2020-01-27,2022-07-12,2481,620,0.511,0.474
4,4,2010-03-16,2022-07-08,2022-07-13,2024-12-27,3101,620,0.508,0.467


OOS accuracy 0.517 vs majority-class baseline 0.565 | OOS AUC 0.488


## 5. Backtest with transaction costs and slippage
Signal on bar *t* -> order filled at the **next bar's open**. `commission_bps` is a relative commission charged on every order; `slippage_bps` is passed as backtesting.py's `spread`, applied to every fill. Total cost per side = commission + slippage (default 10 bps, so about 20 bps per round trip). Long-only: buy when P(up) >= `enter_prob`, exit when P(up) < `exit_prob`.

In [6]:
from backtesting import Backtest, Strategy

class ProbSignalLong(Strategy):
    enter_prob = 0.52
    exit_prob = 0.50
    size_frac = 0.99

    def init(self):
        pass   # the signal is precomputed out-of-sample

    def next(self):
        p = self.data.p_up[-1]
        if not self.position and p >= self.enter_prob:
            self.buy(size=self.size_frac)
        elif self.position and p < self.exit_prob:
            self.position.close()

def run_backtest(prices, p_up, b):
    data = prices.loc[p_up.index, ["Open", "High", "Low", "Close", "Volume"]].copy()
    data["p_up"] = p_up.values
    bt = Backtest(data, ProbSignalLong, cash=b["cash"], commission=b["commission_bps"] / 1e4,
                  spread=b["slippage_bps"] / 1e4, trade_on_close=False, exclusive_orders=True)
    return bt.run(enter_prob=b["enter_prob"], exit_prob=b["exit_prob"], size_frac=b["size_frac"])

stats = run_backtest(prices, oos["p_up"], B)
equity = stats["_equity_curve"]["Equity"]
trades = stats["_trades"]
print(f"{len(trades)} trades | exposure {stats['Exposure Time [%]']:.0f}% | OOS window {equity.index[0].date()} -> {equity.index[-1].date()}")

Backtest.run:   0%|          | 0/3099 [00:00<?, ?bar/s]

369 trades | exposure 82% | OOS window 2012-08-31 -> 2024-12-27


/tmp/ipykernel_3299/3578723672.py:23: UserWarning:

Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.



## 6. Metrics vs buy-and-hold
Strategy and benchmark are scored with the same code. Buy-and-hold pays one entry cost so it is not flattered.

In [7]:
def sharpe_ratio(returns, periods=252, rf_annual=0.0):
    r = returns.dropna() - rf_annual / periods
    sd = r.std(ddof=1)
    return float(np.sqrt(periods) * r.mean() / sd) if sd and sd > 0 else float("nan")

def drawdown_series(equity):
    return equity / equity.cummax() - 1

def summarize(equity, trade_returns=None, periods=252, rf_annual=0.0):
    n = len(equity) - 1
    return {
        "cumulative_return": float(equity.iloc[-1] / equity.iloc[0] - 1),
        "cagr": float((equity.iloc[-1] / equity.iloc[0]) ** (periods / n) - 1),
        "sharpe": sharpe_ratio(equity.pct_change(), periods, rf_annual),
        "max_drawdown": float(drawdown_series(equity).min()),
        "win_rate": float((trade_returns > 0).mean()) if trade_returns is not None and len(trade_returns) else float("nan"),
        "n_trades": int(len(trade_returns)) if trade_returns is not None else 0,
    }

per_side = (B["commission_bps"] + B["slippage_bps"]) / 1e4
bh = B["cash"] * (1 - per_side) * prices.loc[equity.index, "Close"] / prices.loc[equity.index, "Close"].iloc[0]

res = pd.DataFrame({
    "Strategy": summarize(equity, trades["ReturnPct"] if len(trades) else None, B["periods_per_year"], B["risk_free_annual"]),
    "Buy & hold": summarize(bh, None, B["periods_per_year"], B["risk_free_annual"]),
})
display(res.style.format("{:.3f}", na_rep="n/a"))

fig = go.Figure()
fig.add_scatter(x=equity.index, y=equity, name="Strategy")
fig.add_scatter(x=bh.index, y=bh, name="Buy & hold")
fig.update_layout(title="Out-of-sample equity curve", height=380, margin=dict(t=40, b=10))
fig.show()

dd = drawdown_series(equity)
fig = go.Figure(go.Scatter(x=dd.index, y=dd, fill="tozeroy", name="Strategy drawdown"))
fig.update_layout(title="Strategy drawdown", height=260, yaxis_tickformat=".0%", margin=dict(t=40, b=10))
fig.show()

fi = importance.sort_values()
fig = go.Figure(go.Bar(x=fi.values, y=fi.index, orientation="h"))
fig.update_layout(title="Mean feature importance (gain)", height=420, margin=dict(t=40, b=10))
fig.show()

,Strategy,Buy & hold
cumulative_return,0.949,4.288
cagr,0.056,0.145
sharpe,0.460,0.899
max_drawdown,-0.295,-0.337
win_rate,0.564,n/a
n_trades,369.000,0.000


## 7. Sanity checks and leak canary
These guard the results. If any assertion fails, do not trust the numbers above.

In [8]:
# 1) causality: truncating the history must not change earlier feature rows
cut = len(prices) // 2
pd.testing.assert_frame_equal(build_features(prices, F).iloc[:cut], build_features(prices.iloc[:cut], F),
                              check_exact=False, rtol=1e-9)

# 2) label timing: signal at close t -> entry at open t+1 -> exit at open t+1+h
i = 100
expected = prices["Open"].iloc[i + 1 + H] / prices["Open"].iloc[i + 1] - 1
assert np.isclose(tgt["fwd_ret"].iloc[i], expected)

# 3) purge gap between every train block and its test block
assert ((folds["test_start_pos"] - folds["train_end_pos"] - 1) >= H + 1).all()

# 4) the backtest only covers out-of-sample dates
assert equity.index[0] >= oos.index[0] and oos.index[0] > X.index[0]
print("structural checks passed")

# 5) leak canary: a model trained on permuted labels must show no edge (AUC ~ 0.50)
rng = np.random.default_rng(CFG["seed"])
y_perm = pd.Series(rng.permutation(y.values), index=y.index)
_, folds_perm, _ = walk_forward_predict(X, y_perm, fwd, factory, n_splits=cv["n_splits"], gap=H + 1,
                                        test_size=cv["test_size"], max_train_size=cv["max_train_size"])
print(f"mean fold AUC with real labels:     {folds['auc'].mean():.3f}")
print(f"mean fold AUC with shuffled labels: {folds_perm['auc'].mean():.3f}   (should be close to 0.50)")

structural checks passed
mean fold AUC with real labels:     0.487
mean fold AUC with shuffled labels: 0.491   (should be close to 0.50)


## 8. How to read this, and what to do next
- A realistic outcome on a liquid ETF with daily bars is **AUC only slightly above 0.5 and a Sharpe below buy-and-hold after costs**. That is a valid result, not a bug.
- `enter_prob` / `exit_prob` are hyperparameters. Do not tune them on the same out-of-sample window you report; reserve a final holdout that you look at once.
- Adjusted history is revised retroactively by every provider (Yahoo more than most). Keep the saved CSV in `data/` so a run can be reproduced, and re-download only deliberately (`refresh=True`).

**Phase 2 is implemented below, in Part II (sections 9-19):** horizon 5, logistic and momentum baselines, the cost sweep, an LSTM behind the same interface, volatility-regime and lagged macro features, position sizing with a stop-loss and a drawdown halt, rolling-window walk-forward, Optuna scored on inner folds only, SHAP, and a locked holdout evaluated once.

---
# Part II: Phase 2 (the next steps from section 8, implemented)

Everything in sections 10-17 uses a **development window only**. The last `holdout_months` of data are locked away and touched exactly once, in section 18.

| Section | What it does |
|---|---|
| 9 | Phase 2 config and the locked holdout split |
| 10 | Volatility-regime features and macro features from FRED (lagged to publication time) |
| 11 | Logistic regression, a momentum rule and an LSTM behind the same `fit` / `predict_proba` interface |
| 12 | All models side by side, plus expanding vs rolling-window walk-forward |
| 13 | Horizon 1 vs horizon 5 |
| 14 | Cost sweep: 0, 5, 10, 20 bps per side |
| 15 | Risk layer: volatility-targeted sizing, stop-loss, drawdown halt |
| 16 | Optuna tuning scored on inner folds only |
| 17 | SHAP feature attribution |
| 18 | Sanity checks, then the one-shot holdout evaluation |

The LSTM and Optuna are the slow parts. Set `enabled=False` for either in the section 9 config to skip them.

## 9. Phase 2 setup

In [9]:
# torch is preinstalled on Colab. Locally: pip install torch shap optuna
%pip install -q torch shap optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 9.5 MB/s eta 0:00:00


In [10]:
try:
    import torch
    import torch.nn as nn
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAVE_OPTUNA = True
except ImportError:
    HAVE_OPTUNA = False
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False

P2 = dict(
    holdout_months=12,                       # locked until section 18
    cost_grid_bps=[0, 5, 10, 20],            # total cost per side, split half commission / half slippage
    rolling_window=756,                      # about 3 trading years
    macro=dict(enabled=True, series=["VIXCLS", "DGS10"], lag=2),   # FRED, no key needed; lag in trading days
    lstm=dict(enabled=True, seq_len=20, hidden=32, dropout=0.2, epochs=15, batch=64, lr=1e-3, val_frac=0.15),
    optuna=dict(enabled=True, n_trials=20),
    risk=dict(target_vol=0.15, max_frac=0.99, stop_pct=0.05, dd_limit=0.15, cooloff_bars=20),
)
P2["lstm"]["enabled"] = P2["lstm"]["enabled"] and HAVE_TORCH
P2["optuna"]["enabled"] = P2["optuna"]["enabled"] and HAVE_OPTUNA
SEED = CFG["seed"]
if HAVE_TORCH:
    torch.manual_seed(SEED)
print(f"torch: {HAVE_TORCH} | optuna: {HAVE_OPTUNA} | shap: {HAVE_SHAP}")

torch: True | optuna: True | shap: True


## 10. Regime and macro features, and the locked holdout
- **Regime features** come from prices only: short vs long volatility, where today's volatility ranks over the past year, distance from the 252-day high, and distance from the 200-day average.
- **Macro features** come from FRED's public CSV endpoint (no key). VIX (log level) and the 10-year yield, each as a level and a 5-day change. Every macro series is carried forward onto trading days and then **lagged by `lag` trading days** (default 2), because values are published after the date they describe and may be revised. This is deliberately conservative.
- **Holdout:** the final `holdout_months` are split off. The last `horizon + 1` development rows are dropped as well (a purge), so no development label looks into the holdout.

In [11]:
def regime_features(df):
    close = df["Close"]
    ret1 = close.pct_change()
    rv20, rv100 = ret1.rolling(20).std(), ret1.rolling(100).std()
    out = pd.DataFrame(index=df.index)
    out["vol_ratio_20_100"] = rv20 / rv100
    out["vol_pctile_252"] = rv20.rolling(252).rank(pct=True)
    out["dist_from_252_high"] = close / close.rolling(252).max() - 1
    out["close_over_sma200"] = close / close.rolling(200).mean() - 1
    return out.replace([np.inf, -np.inf], np.nan)

def fetch_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    raw = pd.read_csv(url, na_values=["."])
    idx = pd.to_datetime(raw.iloc[:, 0])
    return pd.Series(pd.to_numeric(raw.iloc[:, 1], errors="coerce").to_numpy(), index=pd.DatetimeIndex(idx), name=series_id)

def macro_features(index, series, lag):
    cols = {}
    for s in series:
        x = fetch_fred(s)
        x = x.reindex(x.index.union(index)).ffill().reindex(index).shift(lag)   # last published value, then lagged
        lvl = np.log(x) if s == "VIXCLS" else x
        cols[f"{s.lower()}_lvl"] = lvl
        cols[f"{s.lower()}_chg5"] = lvl.diff(5)
    return pd.DataFrame(cols, index=index)

extra = regime_features(prices)
if P2["macro"]["enabled"]:
    try:
        extra = extra.join(macro_features(prices.index, P2["macro"]["series"], P2["macro"]["lag"]))
    except Exception as e:
        print("macro features skipped (no internet or FRED unavailable):", repr(e)[:160])
BASE_COLS, EXTRA_COLS = list(feats.columns), list(extra.columns)
all_feats = feats.join(extra)

def build_dataset(horizon):
    return all_feats.join(make_target(prices, horizon)).dropna()

def split_dev_holdout(ds, horizon):
    gap = horizon + 1
    cut = ds.index[-1] - pd.DateOffset(months=P2["holdout_months"])
    hold = ds.loc[ds.index > cut]
    dev = ds.loc[ds.index <= cut].iloc[:-gap]       # purge: dev labels must not reach into the holdout
    return dev, hold

ds1 = build_dataset(1)
dev1, hold1 = split_dev_holdout(ds1, 1)
print(f"{len(BASE_COLS)} base + {len(EXTRA_COLS)} extra features")
print(f"dev      {dev1.index[0].date()} -> {dev1.index[-1].date()}  ({len(dev1)} rows)")
print(f"holdout  {hold1.index[0].date()} -> {hold1.index[-1].date()}  ({len(hold1)} rows, locked until section 18)")

17 base + 8 extra features
dev      2011-01-31 -> 2023-12-22  (3247 rows)
holdout  2023-12-28 -> 2024-12-27  (252 rows, locked until section 18)


## 11. More models behind the same interface
Anything with `fit(X, y)` and `predict_proba(X)` plugs into the walk-forward loop.
- **Logistic regression**: a regularized linear baseline. If XGBoost cannot beat it, the trees are not adding value.
- **Momentum rule**: long when the 20-day return is positive. No learning at all; this is the bar a model has to clear.
- **LSTM**: reads the previous `seq_len` days of the same features. Features are standardized with training-fold statistics only, the last 15% of each training window is used for early stopping, and a window for date *t* only ever contains rows up to *t*.

In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

class LogitModel:
    def __init__(self, C=0.1, seed=0):
        self.pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000, random_state=seed))
    def fit(self, X, y):
        self.pipe.fit(X, y)
        return self
    def predict_proba(self, X):
        return self.pipe.predict_proba(X)
    @property
    def feature_importances_(self):
        return np.abs(self.pipe[-1].coef_[0])

class MomentumModel:
    # long when the chosen return column is positive; reported as pseudo-probabilities (0.6 / 0.4)
    def __init__(self, col="ret_20"):
        self.col = col
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        p = np.where(X[self.col].to_numpy() > 0, 0.6, 0.4)
        return np.column_stack([1 - p, p])

if HAVE_TORCH:
    class _LSTMNet(nn.Module):
        def __init__(self, n_feat, hidden, dropout):
            super().__init__()
            self.lstm = nn.LSTM(n_feat, hidden, batch_first=True)
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(self.drop(out[:, -1])).squeeze(-1)

    class LSTMModel:
        # X_all = every feature row that may ever be looked up (windows only reach backwards in time)
        def __init__(self, X_all, cfg, seed=0):
            self.X_all, self.c, self.seed = X_all, cfg, seed
            self.cols = list(X_all.columns)

        def _windows(self, index):
            pos = self.X_all.index.get_indexer(index)
            L = self.c["seq_len"]
            rows = np.clip(pos[:, None] - np.arange(L - 1, -1, -1)[None, :], 0, None)   # (n, L) row positions
            return torch.tensor(self.Z[rows], dtype=torch.float32)

        def fit(self, X, y):
            c = self.c
            torch.manual_seed(self.seed)
            X = X[self.cols]
            mu, sd = X.mean(), X.std().replace(0, 1.0)                      # training-fold statistics only
            self.Z = ((self.X_all[self.cols] - mu) / sd).fillna(0.0).to_numpy(dtype="float64", copy=True)
            xs = self._windows(X.index)
            ys = torch.tensor(y.to_numpy(), dtype=torch.float32)
            n_val = max(1, int(len(xs) * c["val_frac"]))
            xtr, ytr, xva, yva = xs[:-n_val], ys[:-n_val], xs[-n_val:], ys[-n_val:]
            self.net = _LSTMNet(len(self.cols), c["hidden"], c["dropout"])
            opt = torch.optim.Adam(self.net.parameters(), lr=c["lr"], weight_decay=1e-4)
            loss_fn = nn.BCEWithLogitsLoss()
            best, best_state = float("inf"), None
            for _ in range(c["epochs"]):
                self.net.train()
                perm = torch.randperm(len(xtr))
                for i in range(0, len(xtr), c["batch"]):
                    b = perm[i:i + c["batch"]]
                    opt.zero_grad()
                    loss_fn(self.net(xtr[b]), ytr[b]).backward()
                    opt.step()
                self.net.eval()
                with torch.no_grad():
                    v = loss_fn(self.net(xva), yva).item()
                if v < best:
                    best, best_state = v, {k: t.clone() for k, t in self.net.state_dict().items()}
            self.net.load_state_dict(best_state)
            return self

        def predict_proba(self, X):
            self.net.eval()
            with torch.no_grad():
                p = torch.sigmoid(self.net(self._windows(X.index))).numpy()
            return np.column_stack([1 - p, p])
print("models ready | LSTM available:", HAVE_TORCH)

models ready | LSTM available: True


## 12. All models side by side (development window only)
Every variant uses the **same rows, the same purged folds and the same backtest rules**, so the numbers are directly comparable. The rolling-window variant trains on only the most recent `rolling_window` days of each fold.

Read AUC first. A Sharpe gap of a few tenths between models over this many daily bars is mostly noise.

In [13]:
def run_bt(prices, signal, bcfg, strategy=None, extra_cols=None, **params):
    data = prices.loc[
        signal.index, ["Open", "High", "Low", "Close", "Volume"]
    ].copy()

    data["p_up"] = signal.values

    for k, v in (extra_cols or {}).items():
        data[k] = v.reindex(data.index).values

    bt = Backtest(
        data,
        strategy or ProbSignalLong,
        cash=bcfg["cash"],
        commission=bcfg["commission_bps"] / 1e4,
        spread=bcfg["slippage_bps"] / 1e4,
        trade_on_close=False,
        finalize_trades=True,
        exclusive_orders=True
    )

    p = dict(
        enter_prob=bcfg["enter_prob"],
        exit_prob=bcfg["exit_prob"],
        size_frac=bcfg["size_frac"]
    )

    p.update(params)

    return bt.run(**p)

def score(stats, bcfg):
    eq, tr = stats["_equity_curve"]["Equity"], stats["_trades"]
    per_side = (bcfg["commission_bps"] + bcfg["slippage_bps"]) / 1e4
    close = prices.loc[eq.index, "Close"]
    bh = bcfg["cash"] * (1 - per_side) * close / close.iloc[0]
    s = summarize(eq, tr["ReturnPct"] if len(tr) else None, bcfg["periods_per_year"], bcfg["risk_free_annual"])
    b = summarize(bh, None, bcfg["periods_per_year"], bcfg["risk_free_annual"])
    return s, b

cv = CFG["model"]["cv"]
xgb_factory = lambda: make_model(CFG["model"]["params"], SEED)
VARIANTS = {
    "XGBoost (base features)":        dict(cols=BASE_COLS, factory=xgb_factory),
    "XGBoost (+ regime & macro)":     dict(cols=BASE_COLS + EXTRA_COLS, factory=xgb_factory),
    "XGBoost (rolling window)":       dict(cols=BASE_COLS, factory=xgb_factory, max_train=P2["rolling_window"]),
    "Logistic regression":            dict(cols=BASE_COLS, factory=lambda: LogitModel(seed=SEED)),
    "Momentum rule (20d > 0)":        dict(cols=BASE_COLS, factory=lambda: MomentumModel("ret_20")),
}
if P2["lstm"]["enabled"]:
    VARIANTS["LSTM"] = dict(cols=BASE_COLS, factory=lambda: LSTMModel(ds1[BASE_COLS], P2["lstm"], SEED))

def walk_forward_variant(ds, spec, horizon=1):
    return walk_forward_predict(
        ds[spec["cols"]], ds["y"].astype(int), ds["fwd_ret"], spec["factory"],
        n_splits=cv["n_splits"], gap=horizon + 1, test_size=cv["test_size"],
        max_train_size=spec.get("max_train", cv["max_train_size"]))

PRED, IMP, EQ, rows12 = {}, {}, {}, []
for name, spec in VARIANTS.items():
    oos_v, folds_v, imp_v = walk_forward_variant(dev1, spec)
    st = run_bt(prices, oos_v["p_up"], B)
    s, b = score(st, B)
    PRED[name], IMP[name], EQ[name] = oos_v, imp_v, st["_equity_curve"]["Equity"]
    rows12.append({"model": name,
                   "AUC": roc_auc_score(oos_v["y"], oos_v["p_up"]),
                   "accuracy": ((oos_v["p_up"] > 0.5).astype(int) == oos_v["y"]).mean(),
                   "cum_return": s["cumulative_return"], "sharpe": s["sharpe"], "max_drawdown": s["max_drawdown"],
                   "trades": s["n_trades"], "exposure_%": st["Exposure Time [%]"]})
    print("done:", name)

bh_row = {"model": "Buy & hold", "AUC": np.nan, "accuracy": np.nan, "cum_return": b["cumulative_return"],
          "sharpe": b["sharpe"], "max_drawdown": b["max_drawdown"], "trades": 0, "exposure_%": 100.0}
res12 = pd.DataFrame(rows12 + [bh_row]).set_index("model")
display(res12.round(3))

BH_EQ = B["cash"] * (1 - (B["commission_bps"] + B["slippage_bps"]) / 1e4) * prices.loc[next(iter(EQ.values())).index, "Close"] / prices.loc[next(iter(EQ.values())).index, "Close"].iloc[0]
fig = go.Figure()
for name, eq in EQ.items():
    fig.add_scatter(x=eq.index, y=eq, name=name)
fig.add_scatter(x=BH_EQ.index, y=BH_EQ, name="Buy & hold", line=dict(color="black", dash="dot"))
fig.update_layout(title="Development window: out-of-sample equity curves", height=420, margin=dict(t=40, b=10))
fig.show()

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

done: XGBoost (base features)


Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

done: XGBoost (+ regime & macro)


Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

done: Momentum rule (20d > 0)


Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

done: LSTM


,AUC,accuracy,cum_return,sharpe,max_drawdown,trades,exposure_%
model,,,,,,,
XGBoost (base features),0.492,0.515,0.138,0.156,-0.339,341,83.179
XGBoost (+ regime & macro),0.506,0.520,0.399,0.282,-0.295,319,79.926
XGBoost (rolling window),0.481,0.516,0.492,0.340,-0.319,342,82.218
Logistic regression,0.473,0.554,1.484,0.601,-0.334,51,95.305
Momentum rule (20d > 0),0.509,0.531,0.800,0.588,-0.152,124,72.348
LSTM,0.470,0.561,2.083,0.711,-0.335,8,97.634
Buy & hold,NaN,NaN,2.682,0.794,-0.337,0,100.000


## 13. Horizon 1 vs horizon 5
Same model, same rows, same folds; only the label changes (`open[t+1+h] / open[t+1] - 1 > 0`) and the purge gap grows to `h + 1`. A longer horizon trades less often, so costs matter less, but the label is slower to learn.

In [14]:
ds5 = build_dataset(5)
dev5, _ = split_dev_holdout(ds5, 5)
common = dev1.index.intersection(dev5.index)         # identical rows -> identical folds and OOS window
rows13 = []
for h, dsx in [(1, dev1.loc[common]), (5, dev5.loc[common])]:
    oos_h, _, _ = walk_forward_variant(dsx, VARIANTS["XGBoost (base features)"], horizon=h)
    st = run_bt(prices, oos_h["p_up"], B)
    s, b = score(st, B)
    rows13.append({"horizon": h, "AUC": roc_auc_score(oos_h["y"], oos_h["p_up"]),
                   "cum_return": s["cumulative_return"], "sharpe": s["sharpe"], "max_drawdown": s["max_drawdown"],
                   "trades": s["n_trades"], "exposure_%": st["Exposure Time [%]"]})
rows13.append({"horizon": "buy & hold", "AUC": np.nan, "cum_return": b["cumulative_return"], "sharpe": b["sharpe"],
               "max_drawdown": b["max_drawdown"], "trades": 0, "exposure_%": 100.0})
display(pd.DataFrame(rows13).set_index("horizon").round(3))

Backtest.run:   0%|          | 0/2694 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2694 [00:00<?, ?bar/s]

,AUC,cum_return,sharpe,max_drawdown,trades,exposure_%
horizon,,,,,,
1,0.488,0.218,0.198,-0.314,330,83.228
5,0.453,0.638,0.370,-0.335,233,83.970
buy & hold,NaN,2.590,0.782,-0.337,0,100.000


## 14. Cost sweep
Re-runs each model's out-of-sample predictions through the backtester at 0, 5, 10 and 20 bps per side (half commission, half slippage). Where a model's Sharpe falls below buy-and-hold tells you how much cost the signal can survive.

In [15]:
grid = P2["cost_grid_bps"]
sharpe_tbl, ret_tbl, bh_sharpe = {}, {}, []
for name, oos_v in PRED.items():
    sharpe_tbl[name], ret_tbl[name] = [], []
    for cost in grid:
        bc = {**B, "commission_bps": cost / 2, "slippage_bps": cost / 2}
        s, b = score(run_bt(prices, oos_v["p_up"], bc), bc)
        sharpe_tbl[name].append(s["sharpe"]); ret_tbl[name].append(s["cumulative_return"])
for cost in grid:
    bc = {**B, "commission_bps": cost / 2, "slippage_bps": cost / 2}
    bh_sharpe.append(score(run_bt(prices, PRED["XGBoost (base features)"]["p_up"], bc), bc)[1]["sharpe"])

cols_ = [f"{c} bps" for c in grid]
display(pd.concat([pd.DataFrame(sharpe_tbl, index=cols_).T, pd.DataFrame({"buy & hold": bh_sharpe}, index=cols_).T]).round(3))

fig = go.Figure()
for name, vals in sharpe_tbl.items():
    fig.add_scatter(x=grid, y=vals, name=name, mode="lines+markers")
fig.add_scatter(x=grid, y=bh_sharpe, name="Buy & hold", mode="lines+markers", line=dict(color="black", dash="dot"))
fig.update_layout(title="Sharpe vs cost per side (development window)", xaxis_title="bps per side", yaxis_title="Sharpe", height=380, margin=dict(t=40, b=10))
fig.show()

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

,0 bps,5 bps,10 bps,20 bps
XGBoost (base features),0.474,0.315,0.156,-0.161
XGBoost (+ regime & macro),0.571,0.427,0.282,-0.007
XGBoost (rolling window),0.684,0.512,0.340,-0.004
Logistic regression,0.644,0.623,0.601,0.558
Momentum rule (20d > 0),0.756,0.672,0.588,0.419
LSTM,0.718,0.714,0.711,0.704
buy & hold,0.794,0.794,0.794,0.794


## 15. Risk layer
Applied to the base XGBoost signal. Three independent rules, each switchable:
- **Volatility-targeted sizing:** position size = `target_vol / annualized realized vol`, capped at `max_frac`. Calm markets get a bigger position, turbulent ones a smaller one. Size is set at entry.
- **Stop-loss:** an exit `stop_pct` below the entry-day close.
- **Drawdown halt:** if strategy equity falls `dd_limit` below its peak, close the position and stay flat for `cooloff_bars` bars, then restart the drawdown clock from the current equity.

The parameters are set up front and are not tuned on any result below.

In [16]:
class RiskManagedLong(Strategy):
    enter_prob = 0.52
    exit_prob = 0.50
    size_frac = 0.99
    size_mode = "fixed"        # "fixed" | "vol_target"
    target_vol = 0.15
    max_frac = 0.99
    stop_pct = 0.0             # 0 = no stop
    dd_limit = 0.0             # 0 = no drawdown halt
    cooloff_bars = 20

    def init(self):
        self.peak = self.equity
        self.halted_until = -1

    def next(self):
        i, eq = len(self.data), self.equity
        if i < self.halted_until:
            return                                   # cooling off, stay flat
        if i == self.halted_until:
            self.peak = eq                           # cool-off over: restart the drawdown clock
        self.peak = max(self.peak, eq)
        if self.dd_limit and eq / self.peak - 1 <= -self.dd_limit:
            if self.position:
                self.position.close()
            self.halted_until = i + self.cooloff_bars
            return
        p = self.data.p_up[-1]
        if not self.position and p >= self.enter_prob:
            size = self.size_frac
            if self.size_mode == "vol_target":
                ann_vol = self.data.rvol[-1] * np.sqrt(252)
                size = min(self.max_frac, self.target_vol / ann_vol) if ann_vol > 0 else self.max_frac
            size = float(min(0.99, max(0.01, size)))
            if self.stop_pct:
                self.buy(size=size, sl=self.data.Close[-1] * (1 - self.stop_pct))
            else:
                self.buy(size=size)
        elif self.position and p < self.exit_prob:
            self.position.close()

R = P2["risk"]
RVOL = all_feats[f"realized_vol_{F['vol_window']}"]
vt = dict(size_mode="vol_target", target_vol=R["target_vol"], max_frac=R["max_frac"])
RISK_VARIANTS = {
    "No risk layer":       dict(),
    "Vol-targeted sizing": vt,
    "Stop-loss":           dict(stop_pct=R["stop_pct"]),
    "Drawdown halt":       dict(dd_limit=R["dd_limit"], cooloff_bars=R["cooloff_bars"]),
    "All three":           {**vt, "stop_pct": R["stop_pct"], "dd_limit": R["dd_limit"], "cooloff_bars": R["cooloff_bars"]},
}
base_signal = PRED["XGBoost (base features)"]["p_up"]
rows15, EQ15 = [], {}
for name, params in RISK_VARIANTS.items():
    st = run_bt(prices, base_signal, B, strategy=RiskManagedLong, extra_cols={"rvol": RVOL}, **params)
    s, b = score(st, B)
    EQ15[name] = st["_equity_curve"]["Equity"]
    rows15.append({"variant": name, "cum_return": s["cumulative_return"], "sharpe": s["sharpe"],
                   "max_drawdown": s["max_drawdown"], "trades": s["n_trades"], "exposure_%": st["Exposure Time [%]"]})
risk_tbl = pd.DataFrame(rows15).set_index("variant")
display(risk_tbl.round(3))

fig = go.Figure()
for name, eq in EQ15.items():
    fig.add_scatter(x=eq.index, y=eq, name=name)
fig.add_scatter(x=BH_EQ.index, y=BH_EQ, name="Buy & hold", line=dict(color="black", dash="dot"))
fig.update_layout(title="Risk layer variants (development window)", height=400, margin=dict(t=40, b=10))
fig.show()

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2704 [00:00<?, ?bar/s]

,cum_return,sharpe,max_drawdown,trades,exposure_%
variant,,,,,
No risk layer,0.138,0.156,-0.339,341,83.179
Vol-targeted sizing,-0.003,0.057,-0.312,341,83.179
Stop-loss,-0.063,0.029,-0.376,358,83.105
Drawdown halt,-0.113,-0.025,-0.376,334,78.706
All three,-0.121,-0.066,-0.331,347,80.037


## 16. Optuna scored on inner folds only
Tuning on the same data you report is how backtests lie. Here:
1. Hyperparameters are searched using only the **first half of the development window**, scored by mean AUC over 4 inner purged folds.
2. The default and tuned models are then compared **only on out-of-sample dates after that first half**, which the search never saw.

Expect the inner score to beat the later out-of-sample score. That gap is how much of the "improvement" was fitting noise.

In [17]:
TUNED_PARAMS = None
if P2["optuna"]["enabled"]:
    cols16 = BASE_COLS
    n_tune = len(dev1) // 2
    tune = dev1.iloc[:n_tune]
    Xt, yt = tune[cols16], tune["y"].astype(int)

    def inner_auc(params):
        aucs = []
        for tr, te in TimeSeriesSplit(n_splits=4, gap=H + 1).split(Xt):
            m = make_model(params, SEED).fit(Xt.iloc[tr], yt.iloc[tr])
            aucs.append(roc_auc_score(yt.iloc[te], m.predict_proba(Xt.iloc[te])[:, 1]))
        return float(np.mean(aucs))

    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 400, step=50),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
            reg_lambda=trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
            n_jobs=4)
        return inner_auc(params)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=P2["optuna"]["n_trials"])
    TUNED_PARAMS = {**study.best_params, "n_jobs": 4}
    print("best params:", study.best_params)
    print(f"inner-fold AUC: default {inner_auc(CFG['model']['params']):.3f} -> tuned {study.best_value:.3f}  (selected on this data, so optimistic)")

    tune_end = dev1.index[n_tune - 1]
    rows16 = []
    for label, params in [("default", CFG["model"]["params"]), ("tuned", TUNED_PARAMS)]:
        spec = dict(cols=cols16, factory=lambda p=params: make_model(p, SEED))
        oos_t, _, _ = walk_forward_variant(dev1, spec)
        late = oos_t.loc[oos_t.index > tune_end]
        s, b = score(run_bt(prices, late["p_up"], B), B)
        rows16.append({"params": label, "AUC (later half)": roc_auc_score(late["y"], late["p_up"]),
                       "sharpe": s["sharpe"], "cum_return": s["cumulative_return"], "max_drawdown": s["max_drawdown"], "trades": s["n_trades"]})
    rows16.append({"params": "buy & hold", "AUC (later half)": np.nan, "sharpe": b["sharpe"], "cum_return": b["cumulative_return"],
                   "max_drawdown": b["max_drawdown"], "trades": 0})
    display(pd.DataFrame(rows16).set_index("params").round(3))
    VARIANTS["XGBoost (tuned)"] = dict(cols=cols16, factory=lambda: make_model(TUNED_PARAMS, SEED))
else:
    print("Optuna disabled or not installed; skipping.")

best params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.029967309097101588, 'subsample': 0.7824279936868144, 'colsample_bytree': 0.8925879806965068, 'min_child_weight': 4, 'reg_lambda': 1.524996572521356}
inner-fold AUC: default 0.476 -> tuned 0.501  (selected on this data, so optimistic)


Backtest.run:   0%|          | 0/1623 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1623 [00:00<?, ?bar/s]

,AUC (later half),sharpe,cum_return,max_drawdown,trades
params,,,,,
default,0.495,0.180,0.109,-0.339,201
tuned,0.472,0.482,0.586,-0.294,122
buy & hold,NaN,0.708,1.166,-0.337,0


## 17. SHAP
Mean absolute SHAP value per feature (in log-odds of "up"), from a model trained on the first 80% of the development window and explained on the last 20%, which it has not seen. The sign column shows whether higher feature values push the prediction up (+) or down (-). Compare with XGBoost's own gain importance: features that rank high in one and low in the other deserve suspicion.

In [18]:
if HAVE_SHAP:
    cols17 = BASE_COLS + EXTRA_COLS
    k = int(len(dev1) * 0.8)
    tr17, te17 = dev1.iloc[:k - (H + 1)], dev1.iloc[k:]
    m17 = make_model(CFG["model"]["params"], SEED).fit(tr17[cols17], tr17["y"].astype(int))
    sv = shap.TreeExplainer(m17).shap_values(te17[cols17])
    sv = np.asarray(sv[-1] if isinstance(sv, list) else sv)
    with np.errstate(all="ignore"):
        sign = ["+" if np.nan_to_num(np.corrcoef(te17[c_], sv[:, j])[0, 1]) >= 0 else "-" for j, c_ in enumerate(cols17)]
    shap_tbl = pd.DataFrame({"mean_abs_shap": np.abs(sv).mean(axis=0), "sign": sign}, index=cols17).sort_values("mean_abs_shap", ascending=False)
    gain = IMP["XGBoost (+ regime & macro)"].reindex(shap_tbl.index)
    print("Spearman rank agreement between SHAP and gain importance:", round(shap_tbl["mean_abs_shap"].corr(gain, method="spearman"), 3))
    display(shap_tbl.head(12).round(4))
    top = shap_tbl.head(15).iloc[::-1]
    fig = go.Figure(go.Bar(x=top["mean_abs_shap"], y=top.index, orientation="h"))
    fig.update_layout(title="Mean |SHAP| (top 15 features)", height=440, margin=dict(t=40, b=10))
    fig.show()
else:
    print("shap not installed; skipping.")

Spearman rank agreement between SHAP and gain importance: 0.208


,mean_abs_shap,sign
dist_from_252_high,0.1282,-
dgs10_lvl,0.1155,-
vol_pctile_252,0.0775,+
dgs10_chg5,0.0736,+
volume_ratio_20,0.0645,+
macd_hist_norm,0.0573,+
ret_1,0.0548,-
vixcls_chg5,0.0544,+
realized_vol_20,0.0465,-
rsi_14,0.0454,-


## 18. Sanity checks, then the one-shot holdout
**Checks first.** If an assertion fails, do not trust anything above.

**The holdout** is evaluated once. The model and the risk rule are chosen from the **development** tables only (best development Sharpe unless you set `FINAL_MODEL` / `FINAL_RISK`), refit on all development rows, and run on the locked window. If you change a choice after seeing the holdout result, the holdout has become a second development set and its number no longer means anything. The cell refuses to run twice in one session; restart the runtime to repeat it.

In [19]:
# 1) purge between dev and holdout
gap1 = H + 1
pos_dev_end, pos_hold_start = ds1.index.get_loc(dev1.index[-1]), ds1.index.get_loc(hold1.index[0])
assert pos_hold_start - pos_dev_end - 1 >= gap1, "dev labels overlap the holdout"

# 2) no development prediction touches the holdout window
assert all(p.index.max() <= dev1.index[-1] for p in PRED.values())

# 3) macro lag: the feature on date t equals the series as published `lag` trading days earlier
if "vixcls_lvl" in extra.columns:
    raw = fetch_fred("VIXCLS")
    aligned = raw.reindex(raw.index.union(prices.index)).ffill().reindex(prices.index)
    j, lag = 400, P2["macro"]["lag"]
    assert np.isclose(extra["vixcls_lvl"].iloc[j], np.log(aligned.iloc[j - lag])), "macro lag is wrong"

# 4) LSTM causality: changing rows AFTER a date must not change that date's prediction
if P2["lstm"]["enabled"]:
    quick = {**P2["lstm"], "epochs": 1}
    lm = LSTMModel(ds1[BASE_COLS], quick, SEED).fit(dev1[BASE_COLS].iloc[:600], dev1["y"].astype(int).iloc[:600])
    probe = dev1[BASE_COLS].iloc[[300]]
    before = lm.predict_proba(probe)
    lm.Z[ds1.index.get_loc(probe.index[0]) + 1:] += 100.0
    assert np.allclose(before, lm.predict_proba(probe)), "LSTM window leaked future rows"

# 5) feature causality still holds with the extra features (regime features only)
cut_ = len(prices) // 2
pd.testing.assert_frame_equal(regime_features(prices).iloc[:cut_], regime_features(prices.iloc[:cut_]), check_exact=False, rtol=1e-9)
print("Phase 2 sanity checks passed")

Phase 2 sanity checks passed


In [20]:
FINAL_MODEL = None     # None = best development Sharpe among the section 12 models, or set a name from VARIANTS
FINAL_RISK = None      # None = best development Sharpe among the section 15 variants, or set a name from RISK_VARIANTS

if globals().get("HOLDOUT_USED"):
    raise RuntimeError("The holdout was already evaluated in this session. Restart the runtime to repeat it honestly.")
HOLDOUT_USED = True

model_name = FINAL_MODEL or res12.drop(index="Buy & hold")["sharpe"].idxmax()
risk_name = FINAL_RISK or risk_tbl["sharpe"].idxmax()
spec = VARIANTS[model_name]
final = spec["factory"]().fit(dev1[spec["cols"]], dev1["y"].astype(int))
p_hold = pd.Series(final.predict_proba(hold1[spec["cols"]])[:, 1], index=hold1.index)

st = run_bt(prices, p_hold, B, strategy=RiskManagedLong, extra_cols={"rvol": RVOL}, **RISK_VARIANTS[risk_name])
s, b = score(st, B)
eq = st["_equity_curve"]["Equity"]
close_h = prices.loc[eq.index, "Close"]
bh_h = B["cash"] * (1 - (B["commission_bps"] + B["slippage_bps"]) / 1e4) * close_h / close_h.iloc[0]

print(f"Model: {model_name} | risk rule: {risk_name}")
print(f"Holdout window: {hold1.index[0].date()} -> {hold1.index[-1].date()}  ({len(hold1)} trading days)")
print(f"Development Sharpe of this model: {res12.loc[model_name, 'sharpe']:.3f}")
out = pd.DataFrame({"Strategy (holdout)": s, "Buy & hold (holdout)": b})
display(out.style.format("{:.3f}", na_rep="n/a"))

fig = go.Figure()
fig.add_scatter(x=eq.index, y=eq, name="Strategy")
fig.add_scatter(x=bh_h.index, y=bh_h, name="Buy & hold")
fig.update_layout(title="Locked holdout: out-of-sample equity curve", height=360, margin=dict(t=40, b=10))
fig.show()

Backtest.run:   0%|          | 0/251 [00:00<?, ?bar/s]

Model: LSTM | risk rule: No risk layer
Holdout window: 2023-12-28 -> 2024-12-27  (252 trading days)
Development Sharpe of this model: 0.711


,Strategy (holdout),Buy & hold (holdout)
cumulative_return,0.277,0.264
cagr,0.278,0.265
sharpe,2.048,1.942
max_drawdown,-0.083,-0.084
win_rate,1.000,n/a
n_trades,1.000,0.000


## 19. How to read the Phase 2 results
- **Twelve months of daily bars is a small sample.** A holdout Sharpe of 1.5 or -0.5 is compatible with a model that has no edge. Treat the holdout as a smoke alarm (did the development result collapse?), not as proof.
- **Compare to the momentum rule and to logistic regression first.** If XGBoost or the LSTM does not clearly beat them on AUC across folds, the added complexity is not earning its keep.
- **The cost sweep decides practicality.** A signal that only works at 0-5 bps is not a signal you can trade.
- **The gap between Optuna's inner score and its later out-of-sample score** is your estimate of overfitting from tuning.
- Every comparison ran several variants on the same development data, so the best-looking one is biased upward. The more variants you try, the more skeptical you should be of the winner.

**Phase 3 ideas:** a multi-ticker pooled model, deflated Sharpe or a multiple-testing correction across everything tried, experiment tracking (MLflow) and scheduled runs.